# Fetch Drive Revision Timestamps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/fix-worksheet-rename-detection-XrDXm/notebooks/drive-revision-pairs.ipynb)

Fetches the **Drive-level** revision metadata (id, timestamp, author) for
the UBL source spreadsheets and saves the raw JSON to Google Drive.

All analysis happens elsewhere — this notebook just collects the data.

## Sheets

| Sheet | Google Sheet ID |
|-------|-----------------|
| UBL 2.5 Library | `18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY` |
| UBL 2.5 Documents | `1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg` |
| UBL 2.5 Signature | `1T6z2NZ4mc69YllZOXE5TnT5Ey-FlVtaXN1oQ4AIMp7g` |
| UBL 2.4 Library | `1kxlFLz2thJOlvpq2ChRAcv76SiKgEIRtoVRqsZ7OBUs` |
| UBL 2.4 Documents | `1GNpHCS7_QkJtP3QIOdPJWL5N3kQ1EzPznT6M8sPsA0Y` |

In [ ]:
# === Auth ===
from google.colab import auth, drive
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request as AuthRequest

creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
creds.refresh(AuthRequest())
TOKEN = creds.token

drive.mount('/content/drive')
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

In [ ]:
# === Fetch all Drive revisions for each sheet ===
import json, time, datetime
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import HTTPError

SHEETS = {
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
    'ubl25_signature': '1T6z2NZ4mc69YllZOXE5TnT5Ey-FlVtaXN1oQ4AIMp7g',
    'ubl24_library':   '1kxlFLz2thJOlvpq2ChRAcv76SiKgEIRtoVRqsZ7OBUs',
    'ubl24_documents': '1GNpHCS7_QkJtP3QIOdPJWL5N3kQ1EzPznT6M8sPsA0Y',
}


def refresh_token():
    global TOKEN
    if creds.expired:
        creds.refresh(AuthRequest())
        TOKEN = creds.token
    return TOKEN


def api_get(url):
    headers = {'Authorization': f'Bearer {refresh_token()}'}
    for attempt in range(4):
        try:
            with urlopen(Request(url, headers=headers), timeout=30) as r:
                return r.status, json.loads(r.read())
        except HTTPError as e:
            if e.code in (429, 500, 502, 503):
                time.sleep(2 ** (attempt + 1))
                continue
            return e.code, e.read().decode(errors='replace')
    return 0, 'max retries'


def get_revisions(file_id):
    all_revs, page_token = [], None
    while True:
        url = (
            f'https://www.googleapis.com/drive/v3/files/{file_id}/revisions'
            f'?pageSize=1000'
            f'&fields=nextPageToken,revisions(id,modifiedTime,'
            f'lastModifyingUser/displayName,lastModifyingUser/emailAddress,size)'
        )
        if page_token:
            url += f'&pageToken={page_token}'
        status, data = api_get(url)
        if status != 200:
            print(f'  ERROR {status}')
            break
        all_revs.extend(data.get('revisions', []))
        page_token = data.get('nextPageToken')
        if not page_token:
            break
        time.sleep(0.5)
    return all_revs


result = {'fetched_at': datetime.datetime.utcnow().isoformat() + 'Z', 'sheets': {}}

for key, fid in SHEETS.items():
    print(f'{key}...', end=' ')
    revs = get_revisions(fid)
    result['sheets'][key] = {'file_id': fid, 'revision_count': len(revs), 'revisions': revs}
    if revs:
        print(f'{len(revs)} revisions  ({revs[0]["modifiedTime"][:10]} → {revs[-1]["modifiedTime"][:10]})')
    else:
        print('0')
    time.sleep(0.5)

total = sum(s['revision_count'] for s in result['sheets'].values())
print(f'\nTotal: {total} Drive revisions across {len(SHEETS)} sheets')

In [ ]:
# === Save to Drive ===
out_dir = Path('/content/drive/MyDrive/ubl-gc-revisions/drive-revision-pairs')
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / 'drive-revisions-metadata.json'
out_path.write_text(json.dumps(result, indent=2))
print(f'Saved: {out_path}  ({out_path.stat().st_size:,} bytes)')
print(f'\nDownload this file and upload it to the Claude Code session.')